# Validation 1 : k-fold Cross-Validation

## 1- Setup and Import Necessary Libraries

In [1]:
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import pandas as pd

## 2- Loading the Data

First, we'll load the data from the CSV files into pandas DataFrames.

In [2]:
df_tool = pd.read_csv('result_tool.csv')
df_manual = pd.read_csv('manual_labeling.csv')

## 3- Data Preparation

We want to combine the data from the tool and manual labeling. For each code smell, we'll have two columns - one for the tool's results and one for manual labeling.

In [4]:
# Create a new DataFrame based on the tool's data
df_combined = df_tool.copy()

# List of code smells to iterate over
code_smells = [
    'LC', 
    'LEC', 
    'CCC', 
    'LE', 
    'CIC'
]

# Add the data from both the tool and manual labeling for each code smell
for smell in code_smells:
    df_combined[f'{smell}_tool'] = df_tool[smell]
    df_combined[f'{smell}_manual'] = df_manual[smell]

## 4- Initialize KFold and Lists

Set up the KFold cross-validator and the list of code smells.

In [5]:
kf = KFold(n_splits=5)

code_smells = [
    'LC', 
    'LEC', 
    'CCC', 
    'LE', 
    'CIC'
]

## 5- Calculate Metrics for Each Code Smell

For each code smell, use KFold cross-validation to obtain predictions and then compute the precision, recall, f1-score, and accuracy.

In [6]:
# Initialize dictionary to store mean metrics
metrics_mean = {
    'Precision': {},
    'Recall': {},
    'F1-Score': {},
    'Accuracy': {}
}

for smell in code_smells:
    precisions, recalls, f1_scores, accuracies = [], [], [], []
    
    for train_index, test_index in kf.split(df_combined):
        _, df_test = df_combined.iloc[train_index], df_combined.iloc[test_index]
        
        y_true = df_test[f'{smell}_manual']
        y_pred = df_test[f'{smell}_tool']
        
        precisions.append(precision_score(y_true, y_pred))
        recalls.append(recall_score(y_true, y_pred))
        f1_scores.append(f1_score(y_true, y_pred))
        accuracies.append(accuracy_score(y_true, y_pred))
    
    metrics_mean['Precision'][smell] = np.mean(precisions)
    metrics_mean['Recall'][smell] = np.mean(recalls)
    metrics_mean['F1-Score'][smell] = np.mean(f1_scores)
    metrics_mean['Accuracy'][smell] = np.mean(accuracies)

## 6- Convert Metrics Dictionary to DataFrame

Finally, let's convert our metrics dictionary into a pandas DataFrame for easier visualization and analysis.

In [7]:
df_metrics_mean = pd.DataFrame(metrics_mean)

In [8]:
df_metrics_mean

,Precision,Recall,F1-Score,Accuracy
LC,0.956471,0.915842,0.933892,0.976521
LEC,1.000000,0.933333,0.960000,0.997403
CCC,1.000000,1.000000,1.000000,1.000000
LE,1.000000,0.968056,0.983579,0.992174
CIC,0.900000,0.950000,0.904762,0.994771
